# Phase 1 - Basic RAG Pipeline

This notebook is your first **RAG**, or **Retrieval-Augmented Generation**, project. The idea behind RAG is simple but powerful: instead of asking an LLM to answer from memory, you first retrieve relevant evidence from your own document, then give that evidence to the LLM as context.

In Phase 1, you build the retrieval side of the system:

- load the PDF into Python.
- split the PDF text into smaller chunks.
- convert those chunks into embeddings, which are numerical meaning representations.
- store those embeddings in FAISS, a vector database.
- ask a question and retrieve the chunks that look most relevant.

At the end of Phase 1, the system can find evidence, but it does not yet write a polished answer. That is why Phase 2 adds the LLM.


## Step 1 - Install Libraries

Notes:

This cell installs the main building blocks of your RAG pipeline. In beginner projects, package versions matter a lot because LangChain, OpenAI, FAISS, and httpx all depend on each other. A small version mismatch can create errors that look unrelated to your code.

What each package does:

- `langchain`: gives you common RAG abstractions such as document loaders, text splitters, retrievers, prompts, and model wrappers.
- `langchain-community`: contains integrations that live outside the core LangChain package, including `PyPDFLoader` and the FAISS vector store wrapper used here.
- `langchain-core`: contains the shared base classes and interfaces that LangChain packages depend on. It is pinned because `langchain==0.2.16` expects a `0.2.x` core version.
- `langchain-openai`: lets LangChain talk to OpenAI chat models through `ChatOpenAI`. It is pinned to `0.1.23` so it stays compatible with LangChain `0.2.x`.
- `openai`: the official OpenAI Python SDK used underneath `langchain-openai`.
- `httpx`: the HTTP client used by OpenAI. Version `0.27.2` avoids the `Client.__init__() got an unexpected keyword argument proxies` error that happens with this older OpenAI/LangChain combination and newer httpx versions.
- `faiss-cpu`: the vector search engine. It stores embeddings and finds nearby vectors quickly.
- `pypdf`: extracts text from the PDF.
- `sentence-transformers`: provides the local embedding model.

Key lesson: RAG work often has two types of bugs. Logic bugs come from the pipeline design; environment bugs come from package versions. This cell is about making the environment predictable.


In [1]:
!pip install langchain==0.2.16
!pip install langchain-community==0.2.16
!pip install langchain-core==0.2.38
!pip install langchain-openai==0.1.23
!pip install langsmith==0.1.147
!pip install openai==1.40.6
!pip install httpx==0.27.2
!pip install faiss-cpu==1.13.2
!pip install pypdf==4.3.1
!pip install sentence-transformers==3.0.1

Notes:

After changing package versions in a running notebook, restart the kernel before continuing. Python can keep old imports in memory, so a restart gives you a clean environment with the repaired versions.


In [2]:
import importlib.metadata as metadata

packages = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-openai",
    "langsmith",
    "openai",
    "httpx",
]

for package in packages:
    print(f"{package}: {metadata.version(package)}")

langchain: 0.2.16
langchain-core: 0.2.38
langchain-community: 0.2.16
langchain-openai: 0.1.23
langsmith: 0.1.147
openai: 1.40.6
httpx: 0.27.2


## Step 2 - Import Libraries

Notes:

This cell imports the classes you use later. Think of each import as one tool in the RAG assembly line:

- `PyPDFLoader` reads a PDF and turns it into LangChain `Document` objects.
- `RecursiveCharacterTextSplitter` breaks long text into smaller pieces that are easier to search.
- `HuggingFaceEmbeddings` turns text into vectors, which let the computer compare meaning instead of exact words.
- `FAISS` stores those vectors and performs similarity search.

A useful mental model:

- Loader: gets text into the system.
- Splitter: makes text searchable.
- Embedding model: converts text into numbers.
- Vector store: remembers the numbers.
- Retriever: finds the best matching chunks for a question.


In [3]:
# Load PDF documents
from langchain_community.document_loaders import PyPDFLoader

# Split large text into smaller chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create text embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Store embeddings in vector database
from langchain_community.vectorstores import FAISS

In [4]:
!python -m pip install --upgrade pip

## Step 3 - Load the PDF

Notes:

This is the **document loading** step. The PDF on disk is not useful to a RAG system until its text is extracted into a Python object. `PyPDFLoader` reads the file and returns a list called `documents`.

Each item in `documents` is a LangChain `Document`. A `Document` usually has two important parts:

- `page_content`: the text extracted from the page.
- `metadata`: extra information such as the source file path and page number.

Why metadata matters: later, when the system answers a question, you may want to show where the answer came from. Without metadata, the model might answer correctly, but you lose traceability.

One small detail: the filename casing must match the real file name. `Graduate-Visa-Route.pdf` and `Graduate-VISA-Route.pdf` may behave differently on case-sensitive systems.


In [5]:
# Point the loader at the local PDF. Keep the filename casing the same as the file on disk
# so the notebook also works on case-sensitive file systems.
loader = PyPDFLoader("UK-Standard-Visitor-Route.pdf")

# .load() extracts the PDF into LangChain Document objects, usually one per page.
documents = loader.load()

## Step 4 - Check Loaded Content

Notes:

This cell prints the text from the first loaded page. It is a sanity check before you build the rest of the pipeline.

You are checking for three things:

- Did the PDF load successfully?
- Is the extracted text readable?
- Are there artifacts such as page numbers, URLs, print timestamps, headers, or footers?

This matters because the retrieval system can only search what it receives. If the loader extracts noisy text, your vector database will store noisy chunks, and retrieval quality will suffer. In your PDF, you can already see page footer text like GOV.UK print URLs and page numbers. That is not a disaster, but it is something to watch.


In [6]:
# Display first page content
documents[0].page_content

'Visit the UK as a Standard\nVisitor\n1. Overview\nYou can visit the UK as a Standard Visitor for tourism, business, study\n(courses up to 6 months) and other permitted activities.\nYou can usually stay in the UK for up to 6 months. You might be able to\napply to stay for longer in certain circumstances, for example to get medical\ntreatment.\nDepending on your nationality, you may not need a visa to visit the UK.\xa0\nYou should check if you need a visa before you apply.\nWhat you need to do\nWhat you can and cannot do (‘permitted activities’)\nYou can visit the UK as a Standard Visitor:\nfor tourism, for example on a holiday or vacation\nto see your family or friends\nto volunteer for up to 30 days with a registered charityCheck if what you plan to do in the UK is allowed as a Standard\nVisitor.1\nCheck you meet the eligibility requirements. 2\nCheck if you need to apply for a visa to visit the UK. 3\nApply for a Standard Visitor visa online - if you need one. 417/05/2026, 11:06 Prin

## Step 5 - Split Text into Chunks

Notes:

This is the **chunking** step. LLMs and retrievers work better with smaller text units than with one huge document. If you embed an entire PDF as one vector, the meaning becomes too broad. If you split the text into chunks, the retriever can return the specific parts that are relevant to the question.

Your splitter uses:

- `chunk_size=500`: each chunk is roughly 500 characters.
- `chunk_overlap=50`: neighboring chunks share about 50 characters.

The overlap is important. Without overlap, a sentence or idea might be split across two chunks, and neither chunk may contain enough context by itself. Overlap gives the retriever a little cushion.

Chunking tradeoff:

- Smaller chunks are more precise but can lose context.
- Larger chunks preserve context but may retrieve irrelevant extra text.

There is no perfect chunk size. In real RAG projects, chunk size is something you test and tune.


In [7]:
# Create a splitter that breaks long pages into retrieval-sized chunks.
# Overlap preserves a little context across chunk boundaries.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the loaded PDF pages into smaller Document objects.
# The chunk metadata still tracks where each chunk came from.
docs = text_splitter.split_documents(documents)

## Step 6 - Check Number of Chunks

Notes:

This cell tells you how many chunks were created from the PDF. In your run, the document became 50 chunks.

This number helps you understand the size of your retrieval index. A small PDF might create tens of chunks; a large knowledge base might create thousands or millions.

Why this matters:

- Too few chunks can mean each chunk is too broad.
- Too many chunks can mean each chunk is too tiny or noisy.
- The number of chunks affects storage cost, retrieval speed, and answer quality.

For a first RAG project, checking `len(docs)` is a good habit because it confirms the splitter actually did something.


In [8]:
# Number of chunks created
len(docs)

107

## Step 7 - View One Chunk

Notes:

This is a quality check for chunking. Looking at one chunk helps you see what the retriever will later search over.

A good chunk usually contains useful content that could help answer a question. A weak chunk might contain mostly boilerplate, repeated headers, page numbers, URLs, or broken text.

In your notebook, one sample chunk includes footer-like text from the GOV.UK print page. That is a useful discovery. It means the pipeline works, but the source text has some noise. Later, you started cleaning text in Phase 2, which is exactly the kind of improvement RAG projects need.

Beginner lesson: retrieval quality depends heavily on chunk quality. The LLM cannot reliably fix bad retrieval.


In [9]:
# Display first chunk
docs[14].page_content

'your overseas company or organisation\ndeliver training or share knowledge on internal projects with UK\nemployees of the company you work for overseas\ncarry out speciﬁc activities if the overseas company you work for has an\neligible contract of purchase, supply or lease with a UK company or\norganisation - check the rules for ‘Manufacture and supply of goods to the\nUK’ (https://www.gov.uk/guidance/immigration-rules/immigration-rules-appendix-\nvisitor-permitted-activities)\nYou should:'

## Step 8 - Create Embeddings

Notes:

This is the **embedding model** step. Embeddings convert text into vectors, which are lists of numbers that represent meaning. Once text is represented as vectors, similar meanings should be close together in vector space.

You are using `sentence-transformers/all-MiniLM-L6-v2`. This is a lightweight local embedding model. It is a good choice for learning because it does not need an API call and runs relatively quickly.

Example intuition:

- The question `How long can I stay on a Graduate visa?` should be close to chunks about visa duration.
- It does not need to share the exact same words if the meaning is similar.

The warning about `HuggingFaceEmbeddings` being deprecated is not a logic error. It means LangChain moved this class to a newer package path. Your current code can still work, but future projects should consider using `langchain_huggingface`.


In [10]:
# Load a local sentence-transformer embedding model.
# This model turns each chunk and query into comparable numerical vectors.
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/7f/3s2chqx942sfr2r40nb5vhww0000gn/T/ipykernel_8142/1119842075.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/opt/anaconda3/lib/python3.13/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## Step 9 - Store in FAISS Vector Database

Notes:

This cell builds the **vector store**. A vector store is a database designed for similarity search. Instead of asking, `Which text exactly contains this keyword?`, it asks, `Which vectors are closest in meaning to this query vector?`

`FAISS.from_documents(docs, embeddings)` does several things at once:

- takes every chunk in `docs`.
- uses the embedding model to turn each chunk into a vector.
- stores the vector in FAISS.
- keeps the original chunk text and metadata so you can read the matched chunk later.

This is the moment your PDF becomes searchable by meaning. Before this step, you just had text. After this step, you have a semantic search index.


In [11]:
# Build the vector index from your chunks.
# LangChain embeds each chunk, stores the vector in FAISS, and keeps the original text for later retrieval.
vectorstore = FAISS.from_documents(
    docs,
    embeddings
)

## Step 10 - Create Retriever

Notes:

This cell creates a **retriever** from the FAISS vector store. A retriever is the part of the RAG pipeline that accepts a user question and returns relevant documents.

You set `search_kwargs={"k": 4}`. The `k` value controls how many chunks come back for each question.

Why `k` matters:

- If `k` is too low, the retriever might miss useful evidence.
- If `k` is too high, the LLM may receive too much irrelevant context.

For a first project, `k=4` is a sensible starting point. Later you can test `k=3`, `k=5`, or use reranking to improve evidence selection.


In [12]:
# Convert the FAISS store into a retriever interface.
# k controls how many chunks come back for each question.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## Step 11 - Ask Questions

Notes:

This is the first real retrieval test. You define a user question and ask the retriever to find chunks that are semantically similar to it.

Important distinction:

- Retrieval means finding relevant chunks.
- Generation means writing a final answer using those chunks.

At this point, no LLM answer has been generated yet. The retriever is only returning source material.

You use `retriever.invoke(query)`, which is the newer LangChain call style. Older examples may show `get_relevant_documents(query)`, but that method now raises a deprecation warning.


In [13]:
# Define the user question.
query = "What is this document about?"

# Retrieve the most semantically similar chunks from the PDF.
# .invoke() is the newer LangChain retriever API.
retrieved_docs = retriever.invoke(query)

## Step 12 - Display Retrieved Answer

Notes:

This cell prints the top retrieved chunk. The wording says `answer`, but technically this is still retrieved evidence, not a generated answer.

This is an important RAG learning point: the first retrieved chunk is not always the best human answer. Sometimes it may be a footer, URL, or unrelated snippet. That does not mean the whole project failed. It means you need to inspect retrieval quality and improve chunking, cleaning, the query, or the retriever settings.

In a complete RAG system, you usually pass several retrieved chunks to the LLM, not just one. The LLM then writes a natural-language answer grounded in those chunks.


In [14]:
# Display the highest-ranked retrieved chunk.
# If this prints boilerplate like headers or footers, that is a sign to clean the PDF text
# or adjust chunking/retrieval before adding an LLM answer step.
print(retrieved_docs[0].page_content)

https://www.gov.uk/standard-visitor/print 2/29


# Phase 2 - LLM

Phase 2 adds the **generation** part of Retrieval-Augmented Generation.

By this point, you already have retrieval working: the system can search the PDF and return chunks. Now you prepare those chunks for an LLM and ask the LLM to answer using only the retrieved context.

The full RAG flow becomes:

1. User asks a question.
2. Retriever finds relevant chunks from the PDF.
3. Retrieved chunks are joined into a context block.
4. A prompt tells the LLM how to use that context.
5. The LLM generates a final answer.

The goal is not just to get an answer. The goal is to reduce hallucination by grounding the answer in your document.


## Step 1 - Better Chunk Inspection

Notes:

This step prints the first few chunks so you can inspect them more systematically. This is a strong habit. Many RAG problems are easier to solve by looking at the actual chunks instead of guessing.

You are checking:

- Do chunks contain meaningful text?
- Are they too short or too long?
- Are sentence boundaries being broken badly?
- Are headers, footers, URLs, and timestamps appearing too often?
- Would one of these chunks help answer a real user question?

This is like checking your ingredients before cooking. If the chunks are weak, the retrieval and LLM answer will also be weak.


In [15]:
# Inspect first 5 chunks

for i, doc in enumerate(docs[:5]):
    
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:500])


--- Chunk 1 ---
Visit the UK as a Standard
Visitor
1. Overview
You can visit the UK as a Standard Visitor for tourism, business, study
(courses up to 6 months) and other permitted activities.
You can usually stay in the UK for up to 6 months. You might be able to
apply to stay for longer in certain circumstances, for example to get medical
treatment.
Depending on your nationality, you may not need a visa to visit the UK. 
You should check if you need a visa before you apply.
What you need to do

--- Chunk 2 ---
What you need to do
What you can and cannot do (‘permitted activities’)
You can visit the UK as a Standard Visitor:
for tourism, for example on a holiday or vacation
to see your family or friends
to volunteer for up to 30 days with a registered charityCheck if what you plan to do in the UK is allowed as a Standard
Visitor.1
Check you meet the eligibility requirements. 2
Check if you need to apply for a visa to visit the UK. 3

--- Chunk 3 ---
Apply for a Standard Visitor visa o

## Step 2 - Clean Retrieved Chunks

Notes:

This step cleans the text inside each chunk. Your cleaning function replaces newlines with spaces and collapses repeated whitespace.

Why this helps:

- PDF extraction often creates awkward line breaks.
- Newlines can make prompts harder to read.
- Repeated spaces add noise without adding meaning.

What this cleaning does not yet solve:

- It does not remove page numbers.
- It does not remove print timestamps.
- It does not remove repeated URLs or headers.
- It does not repair words broken by PDF extraction.

This is still a useful first cleaning pass. In a future version, you could improve it with regex rules to remove GOV.UK print footers and page markers.


In [16]:
# Function to clean document text

def clean_text(text):
    
    # Remove extra spaces
    text = text.replace('\n', ' ')
    
    # Remove repeated spaces
    text = ' '.join(text.split())
    
    return text

# Clean all chunks

for doc in docs:
    doc.page_content = clean_text(doc.page_content)

#### Retrieve Top 3 Chunks

Notes:

This step retrieves and prints the top chunks again after cleaning. This helps you compare whether the pipeline is returning useful evidence for the question.

Your query is broad: `What is this document about?`. Broad questions can be surprisingly hard for vector search because many chunks may seem loosely related. More specific questions often retrieve better evidence, for example:

- `How long does a Graduate visa last?`
- `Who is eligible for a Graduate visa?`
- `Can you extend a Graduate visa?`

Beginner lesson: retrieval depends on the question. If the query is vague, the retrieved chunks may also feel vague or odd.


In [17]:
# User question
query = "Can you extend your Standard VISA?"

# Retrieve top similar chunks.
# .invoke() is the current LangChain retriever call style.
retrieved_docs = retriever.invoke(query)

# Display retrieved chunks
for i, doc in enumerate(retrieved_docs[:3]):
    
    print(f"\n--- Retrieved Chunk {i+1} ---")
    print(doc.page_content)


--- Retrieved Chunk 1 ---
For example, if you have been in the UK for 3 months, you can apply to
extend your stay for 3 more months. This applies if you needed a visa to
visit the UK and also if you did not need one.
If you need to stay longer for medical treatment
If you’re already in the UK, you can apply to stay for a further 6 months if
you:
have paid for any treatment you’ve already had in the UK
can and will pay the further costs of your treatment

--- Retrieved Chunk 2 ---
continue to meet the medical treatment eligibility requirements (/standard-
visitor/visit-for-medical-reasons)
There is no limit on how many times you can extend your stay. It costs
£1,172 each time you extend.
Documents you must provide
You must get a medical practitioner or NHS consultant who’s registered in
the UK to provide details of your medical condition that needs further
treatment.
If you’re having treatment at an NHS hospital under a reciprocal healthcare

--- Retrieved Chunk 3 ---
There is no limit

## Step 4 - Add OpenAI LLM

Notes:

This step adds the LLM that will write the final answer. In your notebook, `ChatOpenAI` is the LangChain wrapper around an OpenAI chat model.

A few important details:

- You install `openai==1.40.6` and `langchain-openai==0.1.23` because they match the LangChain version used earlier.
- You install `httpx==0.27.2` because newer `httpx` caused the `proxies` initialization error in this environment.
- You should restart the notebook kernel after changing package versions. Otherwise Python may keep old imported packages in memory.
- You should never save a real API key inside a notebook. Use `getpass` or environment variables instead.

The model initialization uses `temperature=0`. Lower temperature makes the model more deterministic, which is usually better for document question answering.


In [18]:
# Keep these versions aligned for this notebook.
# httpx 0.28.x can break openai 1.40.x with: Client.__init__() got an unexpected keyword argument 'proxies'.
!pip install openai==1.40.6
!pip install langchain-openai==0.1.23
!pip install httpx==0.27.2

In [19]:
# ChatOpenAI is the LangChain wrapper around OpenAI chat models.
from langchain_openai import ChatOpenAI

In [ ]:
import os
from getpass import getpass

# Do not save a real API key in the notebook. Type it when the cell runs.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [ ]:
# # Initialize the OpenAI chat model.
# # temperature=0 makes document-QA answers more deterministic.
# llm = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=0
# )

In [ ]:
!pip install transformers accelerate

In [ ]:
from transformers import pipeline

# Load local text generation model
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

## Step 5 - Build Context for the LLM

Notes:

This step combines the retrieved chunks into one context string. The LLM does not automatically know what your retriever found. You have to pass the retrieved text into the prompt.

The line `"\n\n".join(...)` places blank lines between chunks. This makes the context easier for the model to read than one long block of text.

In production RAG systems, this step often becomes more advanced. You may include chunk metadata, page numbers, source links, or only the top-ranked chunks. For this first project, joining `doc.page_content` is the right simple version.


In [ ]:
# Combine retrieved chunks into context

context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

## Step 6 - Create a Grounded Prompt

Notes:

This is the prompt engineering step. The prompt tells the LLM how to behave. In RAG, the most important instruction is that the model should answer using only the retrieved context.

Your prompt has three important parts:

- an instruction: answer only using the context.
- the retrieved context itself.
- the user question.

The fallback instruction, `I could not find the answer in the document.`, is important. It gives the model permission to say it does not know instead of inventing an answer. This is one of the main reasons RAG can reduce hallucinations.


In [ ]:
# Create prompt

prompt = f"""
Answer the question ONLY using the context below.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
'I could not find the answer in the document.'
"""

## Step 7 - Generate the Final Answer

Notes:

This is the generation step. `llm.invoke(prompt)` sends your grounded prompt to the chat model and receives a response. The final printed answer is the first true LLM-generated answer in the notebook.

If this cell fails, the error type tells you where to look:

- Import or validation errors usually mean package versions are mismatched.
- Authentication errors usually mean the API key is missing or invalid.
- Quota or rate limit errors mean your OpenAI account, billing, or usage limit needs attention. The code can be correct while the account still cannot make a request.

In your latest run, the final call reached OpenAI but returned an insufficient quota error. That means model initialization worked; the remaining issue is account quota or billing, not the RAG code structure.


In [ ]:
response = generator(prompt)

print(response[0]['generated_text'])